<a href="https://colab.research.google.com/github/DmitryTheSuslov/AHP_KIS/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
set.seed(123)

simulate_mm1_closed <- function(k, lambda, mu, simulation_time = 10000) {
  # Событийное моделирование замкнутой СМО

  # Состояние: число заявок в системе
  n <- 0

  # Время
  t <- 0

  # Статистика
  total_time <- 0
  total_arrivals <- 0
  total_served <- 0
  time_in_states <- numeric(k + 1)

  # Следующие события
  next_arrival <- rexp(1, k * lambda)  # первое поступление
  next_departure <- Inf

  while (t < simulation_time) {
    # Определяем ближайшее событие
    if (next_arrival < next_departure) {
      # Поступление
      dt <- next_arrival - t
      time_in_states[n + 1] <- time_in_states[n + 1] + dt

      t <- next_arrival
      n <- n + 1
      total_arrivals <- total_arrivals + 1

      # Если сервер был свободен, начинаем обслуживание
      if (n == 1) {
        next_departure <- t + rexp(1, mu)
      }

      # Следующее поступление (если есть свободные программисты)
      if (n < k) {
        next_arrival <- t + rexp(1, (k - n) * lambda)
      } else {
        next_arrival <- Inf  # все программисты ждут
      }

    } else {
      # Завершение обслуживания
      dt <- next_departure - t
      time_in_states[n + 1] <- time_in_states[n + 1] + dt

      t <- next_departure
      n <- n - 1
      total_served <- total_served + 1

      if (n > 0) {
        next_departure <- t + rexp(1, mu)
      } else {
        next_departure <- Inf
      }

      # Теперь есть свободный программист
      if (n < k - 1) {
        next_arrival <- t + rexp(1, (k - n) * lambda)
      }
    }
  }

  # Финальные вероятности (доля времени в каждом состоянии)
  p_sim <- time_in_states / simulation_time

  return(list(
    p = p_sim,
    L = sum((0:k) * p_sim),
    lambda_eff = total_served / simulation_time,
    eta = p_sim[1]
  ))
}

cat("Имитационное моделирование (режим 1)\n")
sim_result <- simulate_mm1_closed(k, lambda, mu, 50000)

cat("Симуляционные вероятности состояний:\n")
cat(round(sim_result$p, 6), "\n")

cat(sprintf("L (теория) = %.4f, L (симуляция) = %.4f\n",
            L_closed, sim_result$L))
cat(sprintf("eta (теория) = %.4f, eta (симуляция) = %.4f\n",
            eta_closed, sim_result$eta))